In [ ]:
from __future__ import annotations

import time

from app.core.config import settings
from app.utils.helpers import generate_id


AI_SERVICE_VERSION = "2026-10-28-embedder-v3"


# ============================================================
# PROCESS-WIDE EMBEDDING MODEL
# ============================================================
# The model used to be created inside every AIService instance, and a
# new AIService is built for every upload and every question. That
# reloaded PyTorch + MiniLM (hundreds of MB, several seconds) each time,
# which is what exhausted memory on small hosts and left uploads stuck
# in "processing". It is now loaded once per process, behind a lock.
#
# Embeddings run on fastembed (ONNX Runtime, no PyTorch): the same
# all-MiniLM-L6-v2 model, 384-dim L2-normalised vectors, so embeddings
# already stored in MongoDB stay valid. PyTorch is no longer a dependency.

import os
import threading

_EMBEDDER = None
_EMBEDDER_LOCK = threading.Lock()
_ENCODE_LOCK = threading.Lock()
_EMBEDDER_INFO = {
    "backend": None,
    "ready": False,
    "error": None,
    "load_seconds": None,
}


class _Embedder:
    """Loaded-once embedding model with a backend-independent encode()."""

    def __init__(self, model_name: str):

        self.backend = "fastembed"
        self.model_name = model_name

        try:
            from fastembed import TextEmbedding
        except ImportError as exc:
            raise RuntimeError(
                "fastembed is not installed. Run: pip install fastembed"
            ) from exc

        self._model = TextEmbedding(
            model_name=model_name,
            cache_dir=os.environ.get("EMBEDDING_CACHE_DIR") or None,
            threads=int(os.environ.get("EMBEDDING_THREADS", "1")),
        )

    def encode(self, texts: list[str], batch_size: int):

        with _ENCODE_LOCK:
            raw = list(
                self._model.embed(
                    texts,
                    batch_size=batch_size,
                )
            )

        vectors = []

        for vector in raw:

            values = [float(value) for value in vector]
            norm = sum(value * value for value in values) ** 0.5

            if norm > 0:
                values = [value / norm for value in values]

            vectors.append(values)

        return vectors


def get_embedder() -> _Embedder:
    """Return the shared embedding model, loading it on first use."""

    global _EMBEDDER

    if _EMBEDDER is not None:
        return _EMBEDDER

    with _EMBEDDER_LOCK:

        if _EMBEDDER is not None:
            return _EMBEDDER

        model_name = getattr(
            settings,
            "embedding_model",
            "sentence-transformers/all-MiniLM-L6-v2",
        )

        started = time.perf_counter()

        try:
            embedder = _Embedder(model_name)

        except Exception as exc:
            message = f"Could not load the embedding model: {exc}"
            _EMBEDDER_INFO.update(ready=False, error=message)
            raise RuntimeError(message) from exc

        _EMBEDDER = embedder

        _EMBEDDER_INFO.update(
            backend=embedder.backend,
            ready=True,
            error=None,
            load_seconds=round(time.perf_counter() - started, 2),
        )

        print(
            f"✓ Embedding model ready: backend={embedder.backend} "
            f"load={_EMBEDDER_INFO['load_seconds']}s"
        )

        return embedder


def embedding_status() -> dict:
    """Non-secret snapshot used by /health."""

    return dict(_EMBEDDER_INFO)


def warm_up_embeddings() -> None:
    """Load the model now so the first upload does not pay for it."""

    try:
        get_embedder().encode(["warm up"], batch_size=1)
    except Exception as exc:
        print(f"! Embedding warm-up failed: {exc}")


class AIService:
    """
    Central AI provider abstraction.

    Text generation:
        Groq

    Embeddings:
        Local sentence-transformer model

    Responsibilities:
        - Groq text generation
        - Six-key API rotation
        - Document embeddings
        - Query embeddings
        - AI usage observability

    The rest of the application communicates with the AI
    layer through this class and does not know which provider
    is being used.
    """

    # ========================================================
    # EMBEDDINGS
    # ========================================================

    EMBEDDING_MODEL = (
        "sentence-transformers/"
        "all-MiniLM-L6-v2"
    )

    EMBEDDING_DIMENSIONS = 384

    # ========================================================
    # INITIALIZATION
    # ========================================================

    def __init__(
        self,
        database=None,
    ):

        self.database = database

        # Index of the key that should be tried first.

        self._current_key_index = 0

    # ========================================================
    # GROQ CLIENT
    # ========================================================

    @staticmethod
    def _create_groq_client(
        api_key: str,
    ):

        try:

            from groq import Groq

        except ImportError as exc:

            raise RuntimeError(
                "groq is not installed. "
                "Run: pip install groq"
            ) from exc

        return Groq(
            api_key=api_key,
            timeout=45.0,
            max_retries=1,
        )

    # ========================================================
    # GET GROQ KEYS
    # ========================================================

    @staticmethod
    def _get_groq_keys() -> list[str]:

        keys = [
            settings.groq_api_key_1,
            settings.groq_api_key_2,
            settings.groq_api_key_3,
            settings.groq_api_key_4,
            settings.groq_api_key_5,
            settings.groq_api_key_6,
        ]

        keys = [
            key.strip()
            for key in keys
            if key and key.strip()
        ]

        if not keys:
            raise RuntimeError(
                "No Groq API keys are configured. "
                "Expected GROQ_API_KEY_1, "
                "GROQ_API_KEY_2, etc."
            )

        return keys

    # ========================================================
    # ROTATION DECISION
    # ========================================================

    @staticmethod
    def _should_rotate_key(
        exc: Exception,
    ) -> bool:
        """
        Return True only when the failure is likely related
        to the provider/key/rate-limit/service availability.

        Normal application errors are NOT rotated.
        """

        status_code = getattr(
            exc,
            "status_code",
            None,
        )

        if status_code in {
            401,
            403,
            429,
            500,
            502,
            503,
            504,
        }:

            return True

        response = getattr(
            exc,
            "response",
            None,
        )

        response_status = getattr(
            response,
            "status_code",
            None,
        )

        if response_status in {
            401,
            403,
            429,
            500,
            502,
            503,
            504,
        }:

            return True

        message = str(
            exc
        ).lower()

        provider_error_markers = (
            "rate limit",
            "rate_limit",
            "too many requests",
            "unauthorized",
            "forbidden",
            "service unavailable",
            "internal server error",
            "bad gateway",
            "gateway timeout",
            "temporarily unavailable",
        )

        return any(
            marker in message
            for marker in provider_error_markers
        )

    # ========================================================
    # TEXT GENERATION
    # ========================================================

    def generate_text(
        self,
        prompt: str,
        system_instruction: str | None = None,
        operation: str = "text_generation",
        user_id: str | None = None,
        project_id: str | None = None,
    ) -> str:

        if not prompt or not prompt.strip():

            raise ValueError(
                "Prompt cannot be empty."
            )

        # Groq has an 8k TPM input limit in the current deployment.
        # Perform a conservative preflight before creating a provider request.
        # Generation-specific services are responsible for trimming/rebuilding
        # their prompts; this check is the final safety gate.
        combined_chars = len(prompt) + len(system_instruction or "")
        approx_input_tokens = (combined_chars + 3) // 4
        safe_input_token_limit = 6000

        if approx_input_tokens > safe_input_token_limit:
            raise ValueError(
                f"{operation} prompt exceeds the safe input budget: "
                f"{combined_chars} characters (~{approx_input_tokens} tokens). "
                f"Maximum safe input is {safe_input_token_limit} tokens."
            )

        print(
            "[AIService] "
            f"operation={operation} "
            f"prompt_chars={combined_chars} "
            f"approx_input_tokens={approx_input_tokens}"
        )

        keys = self._get_groq_keys()

        key_count = len(
            keys
        )

        max_attempts = min(
            max(
                1,
                settings.groq_max_key_attempts,
            ),
            key_count,
        )

        started = time.perf_counter()

        request_id = generate_id()

        last_exception = None

        # ====================================================
        # KEY ROTATION LOOP
        # ====================================================

        for attempt in range(
            max_attempts
        ):

            key_index = (
                self._current_key_index
                + attempt
            ) % key_count

            api_key = keys[
                key_index
            ]

            client = (
                self._create_groq_client(
                    api_key
                )
            )

            try:

                # ------------------------------------------------
                # BUILD MESSAGES
                # ------------------------------------------------

                messages = []

                if system_instruction:

                    messages.append(
                        {
                            "role": "system",
                            "content": (
                                system_instruction
                            ),
                        }
                    )

                messages.append(
                    {
                        "role": "user",
                        "content": prompt,
                    }
                )

                # ------------------------------------------------
                # GROQ REQUEST
                # ------------------------------------------------

                response = (
                    client
                    .chat
                    .completions
                    .create(
                        model=settings.groq_model,
                        messages=messages,
                    )
                )

                # ------------------------------------------------
                # VALIDATE RESPONSE
                # ------------------------------------------------

                if not response.choices:

                    raise RuntimeError(
                        "Groq returned no choices."
                    )

                text = (
                    response
                    .choices[0]
                    .message
                    .content
                )

                if not text:

                    raise RuntimeError(
                        "Groq returned an empty response."
                    )

                # ------------------------------------------------
                # USAGE
                # ------------------------------------------------

                latency_ms = (
                    time.perf_counter()
                    - started
                ) * 1000

                usage = (
                    self._extract_groq_usage(
                        response
                    )
                )

                self._record_usage(
                    operation=operation,
                    user_id=user_id,
                    project_id=project_id,
                    latency_ms=latency_ms,
                    success=True,
                    request_id=request_id,
                    input_tokens=(
                        usage[
                            "input_tokens"
                        ]
                    ),
                    output_tokens=(
                        usage[
                            "output_tokens"
                        ]
                    ),
                    total_tokens=(
                        usage[
                            "total_tokens"
                        ]
                    ),
                )

                # ------------------------------------------------
                # REMEMBER SUCCESSFUL KEY
                # ------------------------------------------------

                self._current_key_index = (
                    key_index
                )

                return text.strip()

            except Exception as exc:

                last_exception = exc

                # ------------------------------------------------
                # DO NOT ROTATE FOR APPLICATION ERRORS
                # ------------------------------------------------

                if not self._should_rotate_key(
                    exc
                ):

                    latency_ms = (
                        time.perf_counter()
                        - started
                    ) * 1000

                    self._record_usage(
                        operation=operation,
                        user_id=user_id,
                        project_id=project_id,
                        latency_ms=latency_ms,
                        success=False,
                        error=str(exc),
                        request_id=request_id,
                    )

                    raise

                # ------------------------------------------------
                # TRY NEXT KEY
                # ------------------------------------------------

                continue

        # ========================================================
        # ALL KEYS FAILED
        # ========================================================

        latency_ms = (
            time.perf_counter()
            - started
        ) * 1000

        self._record_usage(
            operation=operation,
            user_id=user_id,
            project_id=project_id,
            latency_ms=latency_ms,
            success=False,
            error=(
                str(last_exception)
                if last_exception
                else (
                    "All configured Groq "
                    "API keys failed."
                )
            ),
            request_id=request_id,
        )

        if last_exception:

            raise last_exception

        raise RuntimeError(
            "All configured Groq API keys failed."
        )

    # ========================================================
    # GROQ USAGE
    # ========================================================

    @staticmethod
    def _extract_groq_usage(
        response,
    ) -> dict[str, int | None]:

        usage = getattr(
            response,
            "usage",
            None,
        )

        if usage is None:

            return {
                "input_tokens": None,
                "output_tokens": None,
                "total_tokens": None,
            }

        input_tokens = getattr(
            usage,
            "prompt_tokens",
            None,
        )

        output_tokens = getattr(
            usage,
            "completion_tokens",
            None,
        )

        total_tokens = getattr(
            usage,
            "total_tokens",
            None,
        )

        return {
            "input_tokens": (
                int(input_tokens)
                if input_tokens is not None
                else None
            ),
            "output_tokens": (
                int(output_tokens)
                if output_tokens is not None
                else None
            ),
            "total_tokens": (
                int(total_tokens)
                if total_tokens is not None
                else None
            ),
        }

    # ========================================================
    # LOCAL EMBEDDING MODEL
    # ========================================================

    def _get_embedding_model(self):

        # Shared, process-wide model (see get_embedder above).
        return get_embedder()

    # ========================================================
    # DOCUMENT EMBEDDING
    # ========================================================

    def embed_text(
        self,
        text: str,
        task_type: str = "RETRIEVAL_DOCUMENT",
    ) -> list[float]:

        if not text or not text.strip():
            raise ValueError(
                "Cannot embed empty text."
            )

        allowed_task_types = {
            "RETRIEVAL_DOCUMENT",
            "RETRIEVAL_QUERY",
        }

        if task_type not in allowed_task_types:
            raise ValueError(
                "Unsupported embedding task type: "
                f"{task_type}"
            )

        vector = self._get_embedding_model().encode(
            [text],
            batch_size=1,
        )[0]

        # --------------------------------------------------------
        # EMBEDDING DIMENSION VALIDATION
        # --------------------------------------------------------
        # Use the application configuration as the source of
        # truth and normalize it to an integer.
        expected_dimensions = int(
            getattr(
                settings,
                "embedding_dimensions",
                self.EMBEDDING_DIMENSIONS,
            )
        )

        actual_dimensions = len(vector)

        if actual_dimensions != expected_dimensions:
            raise RuntimeError(
                "Unexpected embedding dimension: "
                f"{actual_dimensions}. "
                f"Expected {expected_dimensions}."
            )

        return vector

    # ========================================================
    # BATCH DOCUMENT EMBEDDING
    # ========================================================

    def embed_texts(
        self,
        texts: list[str],
        task_type: str = "RETRIEVAL_DOCUMENT",
        batch_size: int = 2,
    ) -> list[list[float]]:
        """
        Embed many texts with a single (or few) sentence-transformer
        calls instead of one call per text.

        This is what document processing should use: encoding N
        chunks in batches of `batch_size` is dramatically faster
        than N sequential embed_text() calls because the model
        pads/batches its forward pass instead of paying per-call
        overhead N times.
        """

        if not texts:
            return []

        allowed_task_types = {
            "RETRIEVAL_DOCUMENT",
            "RETRIEVAL_QUERY",
        }

        if task_type not in allowed_task_types:
            raise ValueError(
                "Unsupported embedding task type: "
                f"{task_type}"
            )

        cleaned = [
            text.strip() if text else ""
            for text in texts
        ]

        empty_indexes = [
            index
            for index, text in enumerate(cleaned)
            if not text
        ]

        if empty_indexes:
            raise ValueError(
                "Cannot embed empty text at index(es): "
                f"{empty_indexes}"
            )

        vectors = self._get_embedding_model().encode(
            cleaned,
            batch_size=max(1, batch_size),
        )

        expected_dimensions = int(
            getattr(
                settings,
                "embedding_dimensions",
                self.EMBEDDING_DIMENSIONS,
            )
        )

        results = []

        for vector in vectors:

            values = [float(value) for value in vector]

            if len(values) != expected_dimensions:
                raise RuntimeError(
                    "Unexpected embedding dimension: "
                    f"{len(values)}. "
                    f"Expected {expected_dimensions}."
                )

            results.append(values)

        return results

    # ========================================================
    # QUERY EMBEDDING
    # ========================================================

    def embed_query(
        self,
        text: str,
    ) -> list[float]:

        return self.embed_text(
            text=text,
            task_type="RETRIEVAL_QUERY",
        )

    # ========================================================
    # OBSERVABILITY
    # ========================================================

    def _record_usage(
        self,
        operation: str,
        user_id: str | None,
        project_id: str | None,
        latency_ms: float,
        success: bool,
        error: str | None = None,
        request_id: str | None = None,
        input_tokens: int | None = None,
        output_tokens: int | None = None,
        total_tokens: int | None = None,
        retrieval_count: int | None = None,
        citation_count: int | None = None,
        grounded: bool | None = None,
    ) -> None:

        if self.database is None:

            return

        try:

            self.database.collection(
                "ai_usage"
            ).insert_one(
                {
                    "id": generate_id(),
                    "user_id": user_id,
                    "project_id": project_id,
                    "operation": operation,
                    "model": settings.groq_model,
                    "input_tokens": input_tokens,
                    "output_tokens": output_tokens,
                    "total_tokens": total_tokens,
                    "latency_ms": latency_ms,
                    "estimated_cost": None,
                    "success": success,
                    "error": error,
                    "retrieval_count": retrieval_count,
                    "citation_count": citation_count,
                    "grounded": grounded,
                    "request_id": request_id,
                }
            )

        except Exception:

            # Observability must never break
            # a successful AI request.

            pass
